In [6]:
import tarfile
!pwd
def create_model_package(model_files, inference_script, output_path):
    with tarfile.open(output_path, "w:gz") as tar:
        for file in model_files:
            tar.add(file, arcname=f"code/{file}")
        # tar.add(inference_script)# arcname="code/inference.py")

# Usage
model_files = ["blockhouse_mls","inference.py", "requirements.txt"]
inference_script = ""
output_path = "model_report.tar.gz"
create_model_package(model_files, inference_script, output_path)


/home/ec2-user/SageMaker/fastapi_urgent/Blockhouse-ML


In [7]:
import boto3

s3_client = boto3.client('s3')
bucket_name = 'sagemaker-tradingmodel-us-east-1'
s3_key = 'production/version_1/model_report.tar.gz'

s3_client.upload_file(output_path, bucket_name, s3_key)

In [8]:
# ccreate session if local
local = False
import boto3
from sagemaker.local import LocalSession
if local:
    sagemaker_session = LocalSession()
    boto_session = sagemaker_session.boto_session
else:
    boto_session = boto3.Session()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [9]:

def create_sagemaker_model(model_name, role_arn, image_uri, model_data_url):
    sagemaker_client = boto_session.client('sagemaker')
    
    response = sagemaker_client.create_model(
        ModelName=model_name,
        PrimaryContainer={
            'Image': image_uri,
            'ModelDataUrl': model_data_url,
            'Environment': {
                'SAGEMAKER_PROGRAM': 'inference.py',
                'SAGEMAKER_CONTAINER_LOG_LEVEL': '10',
                'PYTHONUNBUFFERED':'1',
                # refernce :https://github.com/aws/sagemaker-python-sdk/issues/2574#issuecomment-1854585409
                'SAGEMAKER_TS_RESPONSE_TIMEOUT': '600', # 10 minutes
                'SAGEMAKER_MODEL_SERVER_TIMEOUT': '600', # 10 minutes
                # 'SAGEMAKER_SUBMIT_DIRECTORY': '/opt/ml'
            }
            
        },
        ExecutionRoleArn=role_arn,
        # EnableNetworkIsolation=False,
        # Mode="SingleModel",

    )
    
    return response

# Usage
model_name = 'model-real-time-inference-report333'
role_arn = "arn:aws:iam::433046797920:role/service-role/AmazonSageMakerServiceCatalogProductsUseRole" #'arn:aws:iam::your-account-id:role/your-sagemaker-role'
image_uri = '433046797920.dkr.ecr.us-east-1.amazonaws.com/blockhouse-ml:deploy-test3'
model_data_url = f's3://{bucket_name}/{s3_key}'

response = create_sagemaker_model(model_name, role_arn, image_uri, model_data_url)
print(f"Model created: {response['ModelArn']}")

Model created: arn:aws:sagemaker:us-east-1:433046797920:model/model-real-time-inference-report333


In [12]:
def create_sagemaker_endpoint_config(model_name,  endpoint_config_name, endpoint_name, role_arn):
    sagemaker_client = boto_session.client('sagemaker')
    
    variant_name = 'variant-1'

    response = sagemaker_client.create_endpoint_config(
        EndpointConfigName=endpoint_config_name,
        ExecutionRoleArn = role_arn,
        ProductionVariants=[
            {
                'InstanceType': 'ml.g4dn.xlarge',
                'InitialInstanceCount': 1,
                "ManagedInstanceScaling": { 
                    "MaxInstanceCount": 1,
                    "MinInstanceCount": 1,
                },
                "VariantName": variant_name,
            }
        ]
    )
    
    sagemaker_client.create_endpoint(
        EndpointName=endpoint_name,
        EndpointConfigName=endpoint_config_name)
    
    response = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
    
    sagemaker_client.create_inference_component(
        InferenceComponentName=model_name+"-inference-component",
        EndpointName=endpoint_name,
        VariantName=variant_name,
        Specification={
            "ModelName": model_name, 
            "ComputeResourceRequirements": { 
                "NumberOfAcceleratorDevicesRequired": 1,
                "NumberOfCpuCoresRequired": 1, 
                "MinMemoryRequiredInMb": 128,
            }
        },
        RuntimeConfig={"CopyCount": 1},
    )
    # response["EndpointStatus"]
    return response

# Usage
model_name = 'model-real-time-inference-report333'
endpoint_config_name = 'endpoint-config-real-time-inference3-report3333'
endpoint_name = 'endpoint-real-time-inference-report3333'

response = create_sagemaker_endpoint_config(model_name, endpoint_config_name, endpoint_name, role_arn)
print(f"Endpoint created: {response['EndpointArn']}")

Endpoint created: arn:aws:sagemaker:us-east-1:433046797920:endpoint/endpoint-real-time-inference-report3333


In [13]:
sagemaker_client = boto_session.client('sagemaker')
response = "Creating"
# while response == "Creating":
response = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
# if response["EndpointStatus"] == 'Creating': print(response["EndpointStatus"])
response = response["EndpointStatus"]
print(response)

Creating


In [35]:
## Run inference from endpoint (AWS) real time inference, this code will be used by front end app
import json
import boto3
# endpoint_name = "Model-1728065209105-deploy-test3-Endpoint-20241005-000648"

# Prepare your JSON payload
payload = {
        "ticker": "SPSK",
        "action": "buy",
    }
request_body = json.dumps(payload)
# Create a low-level client representing Amazon SageMaker Runtime
sagemaker_runtime = boto3.client(
    # credentials resolved by boto3 from the environment (env vars / ~/.aws / IAM role)
    "sagemaker-runtime", region_name='us-east-1',
        )
# Make the prediction

# Gets inference from the model hosted at the specified endpoint:
response = sagemaker_runtime.invoke_endpoint(
    EndpointName="endpoint-real-time-inference-report3333", 
    Body=request_body, #bytes(request_body, 'utf-8')
    ContentType='application/json',
    InferenceComponentName="model-real-time-inference-report333-inference-component"
    )

# Decodes and prints the response body:
print(response['Body'].read().decode('utf-8'))

[{"step": 0, "timestamp": "2024-10-04 15:59:00", "order_type": "Market", "volume": 1, "limit_price": 18.22016422586365}]


In [36]:
import boto3
import json
large_cap_companies =['AAPL', 'CSCO', 'MCD', 'IBM', 'AMZN', 'TSLA', 'PFE', 'MS','MSFT','NVDA']
mid_cap_companies = ['AEG', 'NICE', 'NLY', 'ONTO', 'PSN', 'SAIA', 'OWL','PNW','TWLO','HAS']
small_cap_companies = ['NVAX','AMC','WOLF','IREN','SEDG', 'UPWK','SERV','FSLY','BMBL','ARRY']
actions =['buy', 'sell']
payloads =[]
for action in actions:
    for companies in [large_cap_companies, mid_cap_companies, small_cap_companies]:
        for ticker in companies:
                payload = {
                            "ticker": ticker,
                            "action": action,
                        }
                payloads.append(payload)

# Convert payloads to JSON format and make SageMaker inference requests
sagemaker_runtime = boto3.client("sagemaker-runtime", region_name='us-east-1')
# endpoint_name = "endpoint-real-time-inference3"
# model_name = "your-model-name"  # Replace with actual model name
failed_payloads =[]
# Make inference for each payload
for payload in payloads:
    request_body = json.dumps(payload)
    response = sagemaker_runtime.invoke_endpoint(
                                                EndpointName="endpoint-real-time-inference-report3333", 
                                                Body=request_body, #bytes(request_body, 'utf-8')
                                                ContentType='application/json',
                                                InferenceComponentName="model-real-time-inference-report333-inference-component"
                                                )
    
    # Decode and print the response
    result = response['Body'].read().decode('utf-8')
    print(f"Prediction for {payload['ticker']} ({payload['action']}): {len(result)}")
    if(len(result) == 0):
        failed_payloads.append(payload)
        
print(len(failed_payloads))

Prediction for AAPL (buy): 120
Prediction for CSCO (buy): 119
Prediction for MCD (buy): 120
Prediction for IBM (buy): 120
Prediction for AMZN (buy): 121
Prediction for TSLA (buy): 121
Prediction for PFE (buy): 121
Prediction for MS (buy): 118
Prediction for MSFT (buy): 119
Prediction for NVDA (buy): 121
Prediction for AEG (buy): 120
Prediction for NICE (buy): 120
Prediction for NLY (buy): 121
Prediction for ONTO (buy): 121
Prediction for PSN (buy): 121
Prediction for SAIA (buy): 120
Prediction for OWL (buy): 120
Prediction for PNW (buy): 120
Prediction for TWLO (buy): 119
Prediction for HAS (buy): 120
Prediction for NVAX (buy): 121
Prediction for AMC (buy): 120
Prediction for WOLF (buy): 120
Prediction for IREN (buy): 119
Prediction for SEDG (buy): 119
Prediction for UPWK (buy): 121
Prediction for SERV (buy): 119
Prediction for FSLY (buy): 119
Prediction for BMBL (buy): 120
Prediction for ARRY (buy): 119
Prediction for AAPL (sell): 127
Prediction for CSCO (sell): 125
Prediction for MCD

In [1]:
# # Inference testing locally....
# import inference
# import json
# import os
# model_dir = 'production_test'
# os.makedirs(model_dir, exist_ok=True)

# # Load the models from S3 and local storage
# model = inference.model_fn(model_dir)

# # Simulate a request body
# request_body = json.dumps({
#     "ticker": "AAPL",
#     "action": "sell",
#     "inventory": 5000,
#     "timeframe": 390
# })

# import time
# start_t = time.perf_counter()
# # Run the inference process
# input_data = inference.input_fn(request_body, 'application/json')
# prediction = inference.predict_fn(input_data, model)
# output = inference.output_fn(prediction, 'application/json')
# end_t = time.perf_counter()
# print(f"Time taken: {end_t - start_t}")

# print("Prediction Result:", output)

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


Initialized sell models at production_test\SellEquityModels and production_test\MicroSellEquityModels


INFO:inference:Initialized sell models at production_test\SellEquityModels and production_test\MicroSellEquityModels


Deserializing the input data.


INFO:inference:Deserializing the input data.


...

inside ohlcv
1.7180298999992374
Got the response
LOGGING: Adding Forecasts to data...
defaulting  3458211368924.0
Ticker found, fetching market cap...
Market cap for AAPL: 3448298271600.0
LOGGING: Generating Schedule...
large cap, large scenario model selected
model path for macro  production_test\SellEquityModels/model_large_cap_large.h5
------------------------------------------------Class resetted------------------------------------------------


c:\Users\yashv\miniconda3\envs\MLProj\lib\site-packages\arch\univariate\base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
c:\Users\yashv\miniconda3\envs\MLProj\lib\site-packages\arch\univariate\base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
c:\Users\yashv\miniconda3\envs\MLProj\lib\site-packages\arch\univariate\base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
c:\Users\yashv\miniconda3\envs\MLProj\lib\site-packages\arch\univariate\base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

 

--------------------------------------------------
Timestamp: 2024-10-07 13:30:00+00:00, Action: [ 0.33 50.  ], Shares: 826, Inventory: 4174, TimeLeft: 340
Timestamp: 2024-10-07 14:19:00+00:00, Action: [ 0.32310805 48.30973   ], Shares: 1351, Inventory: 2823, TimeLeft: 291
Timestamp: 2024-10-07 15:09:00+00:00, Action: [ 0.33 50.  ], Shares: 934, Inventory: 1889, TimeLeft: 241
Timestamp: 2024-10-07 15:54:00+00:00, Action: [ 0.33     44.250492], Shares: 625, Inventory: 1264, TimeLeft: 196
Timestamp: 2024-10-07 16:39:00+00:00, Action: [ 0.33     44.686436], Shares: 624, Inventory: 640, TimeLeft: 151
Timestamp: 2024-10-07 17:29:00+00:00, Action: [ 0.3290781 50.       ], Shares: 560, Inventory: 80, TimeLeft: 101
Timestamp: 2024-10-07 18:19:00+00:00, Action: [ 0.33 50.  ], Shares: 505, Inventory: 0, TimeLeft: 51
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
--------------------------------------------------
Steps: 7
Cumulative reward: 0
------

INFO:inference:Inference result: [{'step': 1, 'timestamp': '2024-10-07T13:30:00+00:00', 'order_type': 'Market', 'volume': 826, 'limit_price': 227.04306456305275}, {'step': 2, 'timestamp': '2024-10-07T14:19:00+00:00', 'order_type': 'Limit', 'volume': 1351, 'limit_price': 228.47307950964182}, {'step': 3, 'timestamp': '2024-10-07T15:09:00+00:00', 'order_type': 'Limit', 'volume': 934, 'limit_price': 228.74764426412233}, {'step': 4, 'timestamp': '2024-10-07T15:54:00+00:00', 'order_type': 'Limit', 'volume': 625, 'limit_price': 229.02220901860284}, {'step': 5, 'timestamp': '2024-10-07T16:39:00+00:00', 'order_type': 'Market', 'volume': 624, 'limit_price': 228.09532281375803}, {'step': 6, 'timestamp': '2024-10-07T17:29:00+00:00', 'order_type': 'Market', 'volume': 560, 'limit_price': 228.35838737643437}, {'step': 7, 'timestamp': '2024-10-07T18:19:00+00:00', 'order_type': 'Market', 'volume': 505, 'limit_price': 228.62145193911067}]


Time taken: 9.857343499999843
Prediction Result: [{"step": 1, "timestamp": "2024-10-07T13:30:00+00:00", "order_type": "Market", "volume": 826, "limit_price": 227.04306456305275}, {"step": 2, "timestamp": "2024-10-07T14:19:00+00:00", "order_type": "Limit", "volume": 1351, "limit_price": 228.47307950964182}, {"step": 3, "timestamp": "2024-10-07T15:09:00+00:00", "order_type": "Limit", "volume": 934, "limit_price": 228.74764426412233}, {"step": 4, "timestamp": "2024-10-07T15:54:00+00:00", "order_type": "Limit", "volume": 625, "limit_price": 229.02220901860284}, {"step": 5, "timestamp": "2024-10-07T16:39:00+00:00", "order_type": "Market", "volume": 624, "limit_price": 228.09532281375803}, {"step": 6, "timestamp": "2024-10-07T17:29:00+00:00", "order_type": "Market", "volume": 560, "limit_price": 228.35838737643437}, {"step": 7, "timestamp": "2024-10-07T18:19:00+00:00", "order_type": "Market", "volume": 505, "limit_price": 228.62145193911067}]


In [2]:
from sagemaker import image_uris
image_uris.retrieve(framework='pytorch',region='us-east-1',version='2.2.0',py_version='py310',image_scope='inference', instance_type='ml.g4dn.xlarge')



ValueError: Unsupported Python version: py310. You may need to upgrade your SDK version (pip install -U sagemaker) for newer Python versions. Supported Python version(s): py311.